# Cross-study comparison

Reads the metrics JSONs the four studies have already written. **No model, no GPU** — none of
the runs below need to have happened in this session.

Two things the per-study `summary_table` cannot tell you:

1. **Whether a gap is real.** Two `mean (std)` columns are independent summaries, but both
   models scored the *same* test examples. `paired_bootstrap` joins them by id and resamples
   test examples *and* seeds, so the interval covers the test split and the training noise
   together. On AllSides the seed std is 2.23 while test-sampling contributes about ±1, so an
   interval that ignored seeds would be roughly half as wide as honesty allows.
2. **Whether a result holds up.** With four corpora and ~13 task variants, a sign test is more
   powerful than any single-corpus interval: 12 wins of 13 is p≈0.003. That is the claim being
   made — about the extra pretraining, not about any one dataset.

In [1]:
import sys

sys.path.append(".")

from finetuning import aggregate as agg

## Where the results live

Each root is the directory holding `seed_*`. Folded studies keep the fold one level higher, so
they are listed per fold. Edit to match what you have on disk — missing roots are skipped.

In [2]:
import os

# The metric each study actually reports; comparing them on a common one would misrepresent
# every study but the one it came from.
STUDIES = {
    "allsides":                 ("allsides",                                        "f1_macro"),
    "mitweet/ideology_random":  ("results_mitweet/ideology_indicators_random",       "f1_macro"),
    "mitweet/ideology_facet":   ("results_mitweet/ideology_indicators_facet",        "f1_macro"),
    "mitweet/relevance":        ("results_mitweet/relevance_random",                 "f1_micro"),
    "semeval/stance_taskA":     ("results_semeval/stance_target_taskA",              "f1_favor_against"),
    "semeval/stance_taskB":     ("results_semeval/stance_target_taskB",              "f1_favor_against"),
    "semeval/opinion_taskA":    ("results_semeval/opinion_target_taskA",             "f1_macro"),
    "basil/lexical":            ("results_basil/lexical/fold_0",                     "f1_positive"),
    "basil/informational":      ("results_basil/informational/fold_0",               "f1_positive"),
}

# Every model is measured against this one, so the comparison is paired and directional.
REFERENCE = "politics"

available = {name: spec for name, spec in STUDIES.items() if os.path.isdir(spec[0])}
for name, (root, metric) in STUDIES.items():
    mark = "ok  " if name in available else "MISS"
    print(f"  [{mark}] {name:28s} {root}  ({metric})")
print(f"\n{len(available)} of {len(STUDIES)} roots present")

  [ok  ] allsides                     allsides  (f1_macro)
  [ok  ] mitweet/ideology_random      results_mitweet/ideology_indicators_random  (f1_macro)
  [ok  ] mitweet/ideology_facet       results_mitweet/ideology_indicators_facet  (f1_macro)
  [ok  ] mitweet/relevance            results_mitweet/relevance_random  (f1_micro)
  [MISS] semeval/stance_taskA         results_semeval/stance_target_taskA  (f1_favor_against)
  [MISS] semeval/stance_taskB         results_semeval/stance_target_taskB  (f1_favor_against)
  [MISS] semeval/opinion_taskA        results_semeval/opinion_target_taskA  (f1_macro)
  [MISS] basil/lexical                results_basil/lexical/fold_0  (f1_positive)
  [MISS] basil/informational          results_basil/informational/fold_0  (f1_positive)

4 of 9 roots present


## One study at a time

`pairwise_table` reports `model - reference`. `distinguishable` is whether the 95% interval
excludes zero — that, not the point estimate, is the claim.

In [3]:
STUDY = "allsides"

root, metric = STUDIES[STUDY]
runs = agg.discover_results(root)
print(f"{len(runs)} runs under {root}")
display(agg.coverage(runs))
agg.pairwise_table(runs, reference=REFERENCE, metric=metric, n_boot=2000)

50 runs under allsides


,model,n_seeds,seeds,test_rows
0,politics,5,"1, 13, 42, 1234, 6789",2565
1,tlp_1,5,"1, 13, 42, 1234, 6789",2565
2,tlp_16,5,"1, 13, 42, 1234, 6789",2565
3,tlp_32,5,"1, 13, 42, 1234, 6789",2565
4,tlp_theme_1,5,"1, 13, 42, 1234, 6789",2565
5,tlp_theme_16,5,"1, 13, 42, 1234, 6789",2565
6,tlp_theme_32,5,"1, 13, 42, 1234, 6789",2565
7,tlp_theme_tone_1,5,"1, 13, 42, 1234, 6789",2565
8,tlp_theme_tone_16,5,"1, 13, 42, 1234, 6789",2565
9,tlp_tone_16,5,"1, 13, 42, 1234, 6789",2565


,model,vs,metric,mean_diff,ci_low,ci_high,share_favouring_a,distinguishable,n_examples,n_boot
0,tlp_1,politics,f1_macro,-6.60,-8.92,-4.39,0.000,True,2565,2000
1,tlp_16,politics,f1_macro,-5.36,-8.13,-2.42,0.001,True,2565,2000
2,tlp_32,politics,f1_macro,-5.46,-7.78,-3.14,0.000,True,2565,2000
3,tlp_theme_1,politics,f1_macro,-7.66,-10.49,-5.18,0.000,True,2565,2000
4,tlp_theme_16,politics,f1_macro,-3.67,-6.58,-0.33,0.018,True,2565,2000
5,tlp_theme_32,politics,f1_macro,-7.06,-9.43,-4.73,0.000,True,2565,2000
6,tlp_theme_tone_1,politics,f1_macro,-3.90,-7.46,-0.31,0.015,True,2565,2000
7,tlp_theme_tone_16,politics,f1_macro,-6.87,-9.21,-4.55,0.000,True,2565,2000
8,tlp_tone_16,politics,f1_macro,-6.24,-8.78,-3.71,0.000,True,2565,2000


In [4]:
# The unpaired view, for contrast: these intervals ignore that both models scored the same rows.
agg.summary_table(runs, metrics=["f1_macro", "f1_weighted", "precision_macro", "recall_macro"])

metric,f1_macro,f1_weighted,precision_macro,recall_macro
model,,,,
politics,40.97 (1.40),43.88 (1.41),41.33 (1.50),44.45 (1.87)
tlp_1,34.38 (1.36),38.70 (2.01),34.52 (1.71),35.28 (1.16)
tlp_16,35.62 (2.87),37.77 (2.67),37.06 (3.51),37.58 (3.79)
tlp_32,35.53 (1.77),38.88 (2.46),37.37 (3.18),36.29 (1.58)
tlp_theme_1,33.32 (2.17),37.04 (1.39),33.59 (3.07),34.81 (2.38)
tlp_theme_16,37.29 (3.21),39.73 (2.78),39.02 (5.97),38.95 (2.31)
tlp_theme_32,33.91 (1.77),37.75 (1.45),33.95 (2.40),35.81 (1.04)
tlp_theme_tone_1,37.09 (3.95),41.25 (4.97),37.90 (4.23),40.33 (3.82)
tlp_theme_tone_16,34.09 (1.66),36.47 (1.77),35.65 (1.25),35.34 (3.01)


## Every study at once

One row per (study, model). `n_boot` is deliberately low here — raise it before quoting
anything, the intervals move by a few tenths.

In [5]:
roots = {name: spec[0] for name, spec in available.items()}
metrics = {name: spec[1] for name, spec in available.items()}

table = agg.consistency_table(roots, reference=REFERENCE, metrics=metrics, n_boot=500)
table

,study,model,vs,metric,mean_diff,ci_low,ci_high,distinguishable,n_examples
0,allsides,tlp_1,politics,f1_macro,-6.58,-9.03,-4.33,True,2565
1,allsides,tlp_16,politics,f1_macro,-5.35,-7.95,-2.43,True,2565
2,allsides,tlp_32,politics,f1_macro,-5.43,-7.72,-3.19,True,2565
3,allsides,tlp_theme_1,politics,f1_macro,-7.69,-10.38,-5.19,True,2565
4,allsides,tlp_theme_16,politics,f1_macro,-3.62,-6.50,-0.31,True,2565
5,allsides,tlp_theme_32,politics,f1_macro,-7.10,-9.42,-4.76,True,2565
6,allsides,tlp_theme_tone_1,politics,f1_macro,-3.88,-7.35,-0.41,True,2565
7,allsides,tlp_theme_tone_16,politics,f1_macro,-6.87,-9.19,-4.57,True,2565
8,allsides,tlp_tone_16,politics,f1_macro,-6.20,-8.81,-3.76,True,2565
9,mitweet/ideology_random,tlp_16,politics,f1_macro,1.29,-0.11,2.71,False,2542


## The headline: does the extra pretraining win consistently?

`wins` counts studies where the model beats the reference. The exact two-sided binomial p treats
each study as one trial, which is conservative — the variants within a corpus are not fully
independent.

In [6]:
agg.sign_test(table)

,model,wins,losses,n_studies,mean_diff,sign_test_p
0,tlp_theme_16,2,2,4,-2.01,1.0
1,tlp_16,2,2,4,-2.19,1.0
2,tlp_theme_tone_16,2,2,4,-2.98,1.0
3,tlp_tone_16,2,2,4,-2.99,1.0
4,tlp_theme_tone_1,0,1,1,-3.88,1.0
5,tlp_32,0,1,1,-5.43,1.0
6,tlp_1,0,1,1,-6.58,1.0
7,tlp_theme_32,0,1,1,-7.10,1.0
8,tlp_theme_1,0,1,1,-7.69,1.0


## Folded studies

BASIL runs ten repeated stratified splits, so its folds **overlap** and the across-fold spread is
optimistically narrow — read it as a spread, not as a significance test. AllSides media folds and
MITweet facet folds are true partitions, so `pooled_out_of_fold` applies there and is the better
read-out: a media fold holds out whole outlets, which makes its own test set nearly single-class.

In [7]:
import pandas as pd

BASIL_ROOT, TASK, N_FOLDS = "results_basil", "lexical", 10

rows = []
for fold in range(N_FOLDS):
    path = f"{BASIL_ROOT}/{TASK}/fold_{fold}"
    if not os.path.isdir(path):
        continue
    for run in agg.discover_results(path):
        rows.append({"fold": fold, "model": run.model, "seed": run.seed,
                     "f1_positive": run.metrics.get("f1_positive"),
                     "f1_macro": run.metrics.get("f1_macro")})

if rows:
    frame = pd.DataFrame(rows)
    # Mean over seeds within a fold first, so a model run at more seeds is not weighted higher.
    by_fold = frame.groupby(["model", "fold"])[["f1_positive", "f1_macro"]].mean()
    display(by_fold.groupby("model").agg(["mean", "std", "count"]).round(2))
else:
    print(f"no folds under {BASIL_ROOT}/{TASK}")

no folds under results_basil/lexical


In [8]:
# Pooled out-of-fold, for the partition-style studies. Raises if the folds overlap, which is
# exactly what it should do on BASIL's repeated random splits.
FOLDED = "results_allsides/media_split"

if os.path.isdir(FOLDED):
    runs = agg.discover_folds(FOLDED)
    for model, model_runs in sorted(agg.by_model(runs).items()):
        one_seed = [r for r in model_runs if r.seed == 42]
        print(model, agg.pooled_out_of_fold(one_seed, "f1_macro"))
else:
    print(f"{FOLDED} not present -- AllSides runs on its shipped split by default")

results_allsides/media_split not present -- AllSides runs on its shipped split by default
